In [11]:
!pip install -q \
agno==2.6.2 \
google-genai \
sentence-transformers \
lancedb \
tantivy \
pypdf \
python-dotenv

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/pty.py:95: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  pid, fd = os.forkpty()


In [12]:
import os
from pathlib import Path

from dotenv import load_dotenv

from agno.knowledge.knowledge import Knowledge

from agno.vectordb.lancedb import LanceDb

from agno.knowledge.embedder.sentence_transformer import (
    SentenceTransformerEmbedder
)

from google import genai

In [13]:
load_dotenv()

GOOGLE_API_KEY =""

if GOOGLE_API_KEY is None:
    raise ValueError("GOOGLE_API_KEY not found")

client = genai.Client(
    api_key=GOOGLE_API_KEY
)

print("Gemini Connected")

Gemini Connected


In [14]:
DOCUMENT_FOLDER = "../documents"

VECTOR_DB_PATH = "../vectordb"

In [15]:
pdfs = list(Path(DOCUMENT_FOLDER).glob("*.pdf"))

print(f"Found {len(pdfs)} PDFs")

for pdf in pdfs:
    print("✓", pdf.name)

Found 10 PDFs
✓ Promotion_SOP.pdf
✓ Product_Information.pdf
✓ Retail_Execution_Guide.pdf
✓ Retailer_Matching_Guide.pdf
✓ Assortment_Strategy.pdf
✓ Product_Specifications.pdf
✓ Commercial_Policy.pdf
✓ Retailer_Matching_Framework.pdf
✓ Commercial_FAQs.pdf
✓ Trade_Promotion_Guidelines.pdf


In [16]:
embedder = SentenceTransformerEmbedder(
    id="BAAI/bge-small-en-v1.5"
)

print("Embedder Ready")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedder Ready


In [17]:
vector_db = LanceDb(

    uri=VECTOR_DB_PATH,

    table_name="commercial_knowledge",

    embedder=embedder

)

vector_db.create()

print("Vector DB Ready")

INFO Creating table: commercial_knowledge

Vector DB Ready


[2026-07-02T12:46:14Z WARN  lance::dataset::write::insert] No existing dataset at /home/ec2-user/SageMaker/VP_Dedup/../vectordb/commercial_knowledge.lance, it will be created


In [18]:
knowledge = Knowledge(

    name="Commercial Knowledge",

    description="Commercial Intelligence Knowledge Base",

    vector_db=vector_db,

    max_results=5

)

print("Knowledge Initialized")

Knowledge Initialized


In [19]:
for pdf in pdfs:

    print(f"Ingesting : {pdf.name}")

    knowledge.insert(

        name=pdf.stem,

        path=str(pdf)

    )

print("\nKnowledge Base Created Successfully")

Ingesting : Promotion_SOP.pdf


INFO Adding content from path, 0ccc74b7-63e4-53ec-ab20-199c7b4e2357, Promotion_SOP, ../documents/Promotion_SOP.pdf,
     None

Ingesting : Product_Information.pdf


INFO Adding content from path, cb15a98b-4f92-5601-a510-81a32202b59a, Product_Information,                          
     ../documents/Product_Information.pdf, None

Ingesting : Retail_Execution_Guide.pdf


...

Ingesting : Commercial_FAQs.pdf


INFO Adding content from path, 706ad1c5-6c83-5c33-a5bf-90f885507d9f, Commercial_FAQs,                              
     ../documents/Commercial_FAQs.pdf, None

Ingesting : Trade_Promotion_Guidelines.pdf


INFO Adding content from path, acf5a002-ca71-55ac-8afa-07b3318d73f9, Trade_Promotion_Guidelines,                   
     ../documents/Trade_Promotion_Guidelines.pdf, None


Knowledge Base Created Successfully


In [20]:
print(

    "Documents Indexed :",

    knowledge.vector_db.get_count()

)

Documents Indexed : 10


In [21]:
results = knowledge.search(

    query="What ingredients are present in Pringles Original?"

)

results

INFO Found 5 documents

[Document(content='Product Information Product: Pringles Original Brand: Pringles Category: Salty Snacks Ingredients Dehydrated Potatoes Corn Flour Rice Flour Vegetable Oil Salt Wheat Starch Allergens May contain Milk May contain Wheat Shelf Life 9 Months Storage Store below 30°C Keep away from moisture', id=None, name='Product_Information', meta_data={'page': 1, 'linked_to': 'Commercial Knowledge'}, embedder=SentenceTransformerEmbedder(dimensions=384, enable_batch=False, batch_size=100, id='BAAI/bge-small-en-v1.5', sentence_transformer_client=SentenceTransformer(
   (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
   (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'cls', 'include_prompt': True})
   (2): Normalize({})
 ), prompt=None, normalize_embeddings=False), embedding=[-0.05123143643140793, 0.033

In [22]:
for doc in results:

    print("=" * 80)

    print("Document")

    print(doc.name)

    print()

    print(doc.content[:800])

    print()

Document
Product_Information

Product Information Product: Pringles Original Brand: Pringles Category: Salty Snacks Ingredients Dehydrated Potatoes Corn Flour Rice Flour Vegetable Oil Salt Wheat Starch Allergens May contain Milk May contain Wheat Shelf Life 9 Months Storage Store below 30°C Keep away from moisture

Document
Product_Specifications

Product Specification Manual Product Name : Pringles Original Category : Salty Snacks Brand : Pringles Ingredients Dehydrated Potatoes Vegetable Oil Corn Flour Rice Flour Salt Emulsifier Natural Flavoring Allergens May contain traces of Milk May contain Wheat Gluten Free recipe Storage Conditions Store below 30 degrees Celsius. Avoid direct sunlight. Shelf Life : 12 Months.

Document
Retail_Execution_Guide

Retail Execution Guide Display Standards Eye level shelves preferred. Secondary displays increase visibility. Shelf Availability Target OSA 98% Store Segmentation Gold Stores Silver Stores Bronze Stores Merchandising Premium products Eye L

In [23]:
questions = [

    "What is Promotion ROI?",

    "Explain Weighted Distribution",

    "How does Retailer Matching work?",

    "What allergens are present in Pringles Original?",

    "Explain assortment principles."

]

for q in questions:

    print("=" * 100)

    print(q)

    print()

    docs = knowledge.search(

        query=q

    )

    print(docs[0].content[:500])

    print()

What is Promotion ROI?



INFO Found 5 documents

Trade Promotion Guidelines Trade Promotion Guidelines Discounts should normally remain between 10% and 20%. Promotions should run for a maximum of four weeks. Evaluate promotion ROI after campaign completion. Monitor cannibalization across nearby SKUs. Prefer feature displays during launch periods. Checkout displays generally improve conversion. Review incremental sales rather than gross sales. Measure uplift against historical baseline.

Explain Weighted Distribution



INFO Found 5 documents

Commercial FAQs Commercial FAQs What is Numeric Distribution? Numeric Distribution measures store availability. What is Weighted Distribution? Weighted Distribution measures sales weighted availability. What is Promotion ROI? Promotion ROI = Incremental Margin / Promotion Spend. What is Assortment Optimization? Selecting the best products for each store cluster. What is Retailer Matching? Linking retailer records across multiple enterprise systems.

How does Retailer Matching work?



...

Retailer Matching Guidelines Purpose Retailer Matching aligns retailer records across multiple commercial systems. Matching Signals Retailer Name Similarity Address Similarity PIN Code Latitude Longitude Store Type Confidence Score Above 95% Automatic Match 80-95% Manual Review Below 80% Reject Business Rule Never overwrite an approved retailer match. Always maintain audit trail.

What allergens are present in Pringles Original?



INFO Found 5 documents

Product Information Product: Pringles Original Brand: Pringles Category: Salty Snacks Ingredients Dehydrated Potatoes Corn Flour Rice Flour Vegetable Oil Salt Wheat Starch Allergens May contain Milk May contain Wheat Shelf Life 9 Months Storage Store below 30°C Keep away from moisture

Explain assortment principles.



INFO Found 5 documents

Assortment Optimization Strategy Purpose The assortment strategy defines which SKUs should be retained, expanded or discontinued. Decision Factors Revenue Distribution Velocity Margin Demand Transfer Strategic Importance SKU Rationalization Products contributing less than 1% category revenue for three consecutive quarters should be reviewed. Portfolio Expansion High growth SKUs High velocity stores Premium category



In [24]:
print("""
=========================================

Commercial Knowledge Base Ready

Vector Database : LanceDB

Knowledge API : Agno

Embedding : BAAI/bge-small-en-v1.5

Documents : 5

Ready for Notebook 03

=========================================
""")



Commercial Knowledge Base Ready

Vector Database : LanceDB

Knowledge API : Agno

Embedding : BAAI/bge-small-en-v1.5

Documents : 5

Ready for Notebook 03


